# Soup Kernel Challenge: lab runs (A/B of our options + profiler)

Not the submission notebook: it trains only our solution in a few configurations and prints wall time, peak memory, train loop and the top CUDA kernels.

In [ ]:
import os, subprocess
gpus = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                      capture_output=True, text=True).stdout
print(gpus)
assert "T4" in gpus, "Switch the accelerator to GPU T4 x2"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
!python -m pip install -q "soup-cli[train]==0.77.0" "transformers==5.16.1" "peft==0.20.0" "trl==0.29.1" "bitsandbytes==0.50.2" "accelerate==1.14.0" pytest

In [ ]:
!rm -rf /kaggle/working/repo && git clone -q -b claude/v4 https://github.com/tokha-t/decik.git /kaggle/working/repo && cd /kaggle/working/repo && git log --oneline -1
!cd /kaggle/working/repo && python task/prepare_data.py
!cd /kaggle/working/repo && python -m pytest solution -q 2>&1 | tail -15

In [ ]:
import json, os, subprocess
REPO = "/kaggle/working/repo"
summary = {}
def run(tag, profile=False, **env):
    cmd = ["python", "harness/run_task.py", "--only", "solution", "--out", f"runs_{tag}"] + (["--profile"] if profile else [])
    proc = subprocess.run(cmd, cwd=REPO, env=dict(os.environ, **env), capture_output=True, text=True)
    print(proc.stdout[-1500:])
    if proc.returncode:
        print(proc.stderr[-3000:])
    try:
        r = json.load(open(f"{REPO}/runs_{tag}/result.json"))
        s = r["runs"]["solution"][0]
        m = s["metrics"]
        line = dict(tag=tag, env=env, wall_s=s["wall_s"], peak_mib=s["gpu_peak_mib"], loop_s=m.get("train_loop_s"),
                    median_step_s=m.get("median_step_s"), apply_s=m.get("apply_s"), first_step_s=None,
                    eval_loss=r["evals"].get("solution", {}).get("mean_loss"), reserved=m.get("torch_max_reserved_mib"),
                    own_ms={k: round(v) for k, v in (m.get("own_kernel_ms") or {}).items()})
        summary[tag] = line
        print("RESULT", json.dumps(line))
    except Exception as exc:
        print("no result", tag, exc)
        for log in sorted(__import__("glob").glob(f"{REPO}/runs_{tag}/*/train.log")):
            print(log); print(open(log).read()[-4000:])

BASE = ["python", "harness/run_task.py", "--only", "base", "--out", "runs_base"]
print(subprocess.run(BASE, cwd=REPO, capture_output=True, text=True).stdout[-600:])
run("v4", profile=True)
run("v4rc", SKC_RECOMPUTE="1")
run("v3eq", SKC_ATTN_RECOMPUTE="0", SKC_LORA_RECOMPUTE="0")
print("\nSUMMARY")
for tag, line in summary.items():
    print("RESULT", json.dumps(line))

In [ ]:
import json
p = "/kaggle/working/repo/runs_v4/solution_profile/profile.json"
prof = json.load(open(p))
steps = prof["active_steps"]
total = prof["total_gpu_us"] / steps / 1000
print(f"GPU time per step: {total:.1f} ms over {steps} steps")
for row in prof["kernels"][:40]:
    print(f"{row['time_us'] / steps / 1000:8.2f} ms/step  {100 * row['time_us'] / prof['total_gpu_us']:5.1f}%  x{row['count'] // steps:<5d} {row['name'][:110]}")